

<div style="background: linear-gradient(120deg, #1a3a5c 0%, #2d6a9f 60%, #4a9eda 100%); padding: 28px 36px; border-radius: 14px; display: flex; align-items: center; gap: 28px; box-shadow: 0 4px 18px rgba(0,0,0,0.18);">
    <img src='Figures/iteso.jpg' style="height: 110px; border-radius: 8px; background: white; padding: 6px; flex-shrink: 0; box-shadow: 0 2px 8px rgba(0,0,0,0.2);"/>
    <div style="border-left: 2px solid rgba(255,255,255,0.4); padding-left: 28px;">
        <h1 style="margin: 0 0 8px 0; color: white; font-size: 1.5em; line-height: 1.3;">Ingeniería y Ciencia de Datos</h1>
        <h3 style="margin: 0 0 8px 0; color: white; font-size: 1.5em; line-height: 1.3;">Laboratorio de Procesamiento de Datos</h3>
        <h3 style="margin: 0; color: rgba(255,255,255,0.8); font-weight: normal; font-size: 1.05em;">Módulo 1: Comprension de los Datos</h3>
    </div>
</div>



**En esta unidad vamos a tratar los siguientes puntos:**

1. Distinguir variables cualitativas, cuantitativas, binarias, nominales y ordinales.
2. Reconocer la diferencia entre el tipo almacenado (`dtype`) y la escala de medición.
3. Diagnosticar variables en un conjunto de datos real con `pandas`.
4. Identificar valores faltantes y elegir una estrategia con criterio.



# 1. Clasificación de Tipos de Datos
## 1.1 ¿Qué es un dato y por qué importa conocer su tipo?

Un **dato** es una representación de una característica de una entidad, observación o evento. En un dataframe, cada columna representa una variable y cada fila una observación. La taxonomía ayuda a responder preguntas prácticas:

- ¿Tiene sentido calcular un promedio?
- ¿Existe un orden entre las categorías?
- ¿Qué codificación necesita el modelo?
- ¿Qué valores representan ausencia, desconocido o una categoría real?

**Dos perspectivas que conviene separar**

| Perspectiva | Pregunta | Ejemplo |
|---|---|---|
| **Tipo almacenado** | ¿Cómo está guardado en Python? | `int64`, `float64`, `object`, `bool` |
| **Significado estadístico** | ¿Qué operaciones representan la realidad? | nominal, ordinal, discreta, continua |

Una columna guardada como número no necesariamente es cuantitativa. Por ejemplo, `1 = rojo`, `2 = azul` sigue siendo **nominal**: los números son etiquetas y no cantidades.

### Escalas y tipos de variable

| Tipo | Característica | Ejemplos | Operaciones razonables |
|---|---|---|---|
| **Binaria** | Dos estados | `yes/no`, `0/1`, presencia/ausencia | proporción, conteo, tasa |
| **Nominal** | Categorías sin orden | trabajo, país, color | frecuencia, moda |
| **Ordinal** | Categorías con orden, sin distancias necesariamente iguales | primaria/secundaria/terciaria, bajo/medio/alto | comparación de orden, mediana |
| **Cuantitativa discreta** | Conteos enteros | número de contactos, hijos, compras | suma, promedio, dispersión |
| **Cuantitativa continua** | Mediciones en una escala | peso, duración, temperatura | operaciones aritméticas y dispersión |


**Clasificar una variable no es adivinar: es seguir una secuencia de preguntas que el código puede responder.**

Esta secuencia es exactamente la que aplicaremos con el archivo `bank.csv`: 

- primero un diagnóstico general (`info`, `nunique`), después separación por tipo almacenado (`select_dtypes`) y finalmente el significado de cada columna (binaria, nominal, ordinal).
- Al final del cuaderno convertiremos esta secuencia en una **función reutilizable** que genera un reporte de calidad y clasificación para cualquier `DataFrame`.

| Paso | Pregunta | Código en `pandas` | Qué nos dice |
|---|---|---|---|
| 1 | ¿Cómo lo guardó pandas? | `df.dtypes`, `df.info()` | Tipo almacenado (`int64`, `float64`, `object`, `bool`, `category`) |
| 2 | ¿Cuántas categorías distintas tiene? | `df[col].nunique()` | Pista de si es binaria (2), categórica (pocas) o posiblemente continua (muchas) |
| 3 | ¿Cuáles son esos valores? | `df[col].unique()`, `df[col].value_counts()` | Si son etiquetas de texto, códigos numéricos disfrazados o una escala real |
4 | ¿Existe un orden lógico entre las categorías? | Revisión manual + `pd.CategoricalDtype(ordered=True)` | Nominal (sin orden) vs. ordinal (con orden) |
| 5 | Si es numérica, ¿son conteos o mediciones continuas? | `df[col].agg(['min','max','nunique'])` | Discreta (enteros, rango acotado) vs. continua (decimales, muchos valores) |
| 6 | ¿Hay valores faltantes que puedan sesgar la clasificación? | `df[col].isna().sum()` | Evita confundir una categoría real con datos ausentes |



Referencias: 
- [pandas `select_dtypes`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.select_dtypes.html)
- [pandas `CategoricalDtype`](https://pandas.pydata.org/docs/user_guide/categorical.html)
- [scikit-learn: encoding categorical features](https://scikit-learn.org/stable/auto_examples/compose/plot_column_transformer_mixed_types.html)


###  Caso de estudio: campañas de marketing bancario

Trabajaremos con `bank.csv`. Cada fila representa un contacto de una campaña telefónica y `y` indica si la persona contrató el producto ofrecido.

Antes de ejecutar el código, tratemos de deducir la clasificación de estas variables:

- `age`, `balance`, `duration`
- `job`, `marital`, `education`, `month`
- `default`, `housing`, `loan`, `y`

La predicción importa: la clasificación que hagas guiará la exploración y la transformación posterior.

In [1]:
import pandas as pd

In [2]:
df_bank = pd.read_csv('Data/bank.csv')

In [3]:
df_bank.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,30,unemployed,married,primary,no,1787,no,no,cellular,19,oct,79,1,-1,0,unknown,no
1,33,services,married,secondary,no,4789,yes,yes,cellular,11,may,220,1,339,4,failure,no
2,35,management,single,tertiary,no,1350,yes,no,cellular,16,apr,185,1,330,1,failure,no
3,30,management,married,tertiary,no,1476,yes,yes,unknown,3,jun,199,4,-1,0,unknown,no
4,59,blue-collar,married,secondary,no,0,yes,no,unknown,5,may,226,1,-1,0,unknown,no


In [4]:
df_bank.tail()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
4516,33,services,married,secondary,no,-333,yes,no,cellular,30,jul,329,5,-1,0,unknown,no
4517,57,self-employed,married,tertiary,yes,-3313,yes,yes,unknown,9,may,153,1,-1,0,unknown,no
4518,57,technician,married,secondary,no,295,no,no,cellular,19,aug,151,11,-1,0,unknown,no
4519,28,blue-collar,married,secondary,no,1137,no,no,cellular,6,feb,129,4,211,3,other,no
4520,44,entrepreneur,single,tertiary,no,1136,yes,yes,cellular,3,apr,345,2,249,7,other,no


In [5]:
df_bank.shape

(4521, 17)

In [6]:
df_bank.columns

Index(['age', 'job', 'marital', 'education', 'default', 'balance', 'housing',
       'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays',
       'previous', 'poutcome', 'y'],
      dtype='object')

### Lectura rápida 

`info()` permite revisar tamaño, tipos almacenados y valores no nulos. Esta salida es un diagnóstico inicial, no una clasificación completa: una variable `object` puede contener categorías, texto libre o fechas mal interpretadas.

En la siguiente tabla verificaremos qué tipo de dato observa `pandas` en cada columna, cuántos valores distintos tiene (`nunique`) y cuántos faltantes hay. Estas tres columnas ya permiten anticipar la clasificación:

- **`valores_unicos` alto y `tipo_python` numérico** ---> candidata a **cuantitativa** (falta decidir si discreta o continua).

- **`valores_unicos` muy bajo (2)** ---> candidata a variable **binaria**.

- **`valores_unicos` bajo y `tipo_python == object`** ---> candidata a **categórica** (falta decidir si nominal u ordinal).

In [7]:
df_bank.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4521 entries, 0 to 4520
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        4521 non-null   int64 
 1   job        4521 non-null   object
 2   marital    4521 non-null   object
 3   education  4521 non-null   object
 4   default    4521 non-null   object
 5   balance    4521 non-null   int64 
 6   housing    4521 non-null   object
 7   loan       4521 non-null   object
 8   contact    4521 non-null   object
 9   day        4521 non-null   int64 
 10  month      4521 non-null   object
 11  duration   4521 non-null   int64 
 12  campaign   4521 non-null   int64 
 13  pdays      4521 non-null   int64 
 14  previous   4521 non-null   int64 
 15  poutcome   4521 non-null   object
 16  y          4521 non-null   object
dtypes: int64(7), object(10)
memory usage: 600.6+ KB


In [8]:
schema_bank = pd.DataFrame({
    'tipo_python': df_bank.dtypes.astype(str),
    'valores_unicos': df_bank.nunique(),
})
schema_bank

,tipo_python,valores_unicos
age,int64,67
job,object,12
marital,object,3
education,object,4
default,object,2
balance,int64,2353
housing,object,2
loan,object,2
contact,object,3
day,int64,31


## 1.2 Variables cualitativas y cuantitativas

`select_dtypes()` clasifica por el tipo almacenado, lo que es útil como primer filtro: separa en un solo paso las columnas numéricas (`include='number'`) de las que pandas interpreta como texto u objeto (`include='object'`). 

Esto es rápido, pero **no** es la clasificación final: después debemos revisar el significado de cada columna y documentar excepciones (por ejemplo, un código numérico que en realidad es una etiqueta nominal).

In [9]:
df_bank.head(2)

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,30,unemployed,married,primary,no,1787,no,no,cellular,19,oct,79,1,-1,0,unknown,no
1,33,services,married,secondary,no,4789,yes,yes,cellular,11,may,220,1,339,4,failure,no


In [10]:
# Obteniendo datos cuantitativos
df_bank_cuantitativos = df_bank.select_dtypes(include='number')
df_bank_cuantitativos.head()

,age,balance,day,duration,campaign,pdays,previous
0,30,1787,19,79,1,-1,0
1,33,4789,11,220,1,339,4
2,35,1350,16,185,1,330,1
3,30,1476,3,199,4,-1,0
4,59,0,5,226,1,-1,0


In [11]:
df_bank_cuantitativos.columns

Index(['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous'], dtype='object')

## 1.3 Cuantitativas discretas y continuas

La separación `number` / `object` no basta. En este dataset, `age`, `balance` y `duration` son medidas cuantitativas; `campaign`, `pdays` y `previous` son conteos o códigos numéricos cuyo significado merece una revisión adicional.

Una heurística práctica para distinguirlas con código:

Podemos observar sus rangos y valores únicos antes de elegir una transformación:

- Si el `dtype` es entero (`int64`) **y** el número de valores únicos es pequeño en relación al total de filas, suele tratarse de un **conteo discreto** (p. ej. `campaign`, `previous`).
- 
- Si el `dtype` es decimal (`float64`) o el número de valores únicos es muy alto, suele tratarse de una **medición continua** (p. ej. `balance`, `duration`).

In [12]:
df_bank_cuantitativos.head()

,age,balance,day,duration,campaign,pdays,previous
0,30,1787,19,79,1,-1,0
1,33,4789,11,220,1,339,4
2,35,1350,16,185,1,330,1
3,30,1476,3,199,4,-1,0
4,59,0,5,226,1,-1,0


In [13]:
columnas_revisar = ['age', 'balance', 'duration', 'campaign', 'pdays', 'previous']
df_bank[columnas_revisar].agg(['min', 'max', 'nunique']).T

,min,max,nunique
age,19,87,67
balance,-3313,71188,2353
duration,4,3025,875
campaign,1,50,32
pdays,-1,871,292
previous,0,25,24


In [14]:
#Obteniendo datos cualitativos
df_bank_categoricos = df_bank.select_dtypes(exclude='number')
df_bank_categoricos.head()

,job,marital,education,default,housing,loan,contact,month,poutcome,y
0,unemployed,married,primary,no,no,no,cellular,oct,unknown,no
1,services,married,secondary,no,yes,yes,cellular,may,failure,no
2,management,single,tertiary,no,yes,no,cellular,apr,failure,no
3,management,married,tertiary,no,yes,yes,unknown,jun,unknown,no
4,blue-collar,married,secondary,no,yes,no,unknown,may,unknown,no


In [15]:
df_bank_categoricos.columns

Index(['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact',
       'month', 'poutcome', 'y'],
      dtype='object')

## 1.4 Variables binarias

Las columnas `default`, `housing`, `loan` y `y` tienen dos categorías. La codificación `yes/no` conserva legibilidad.

>En código, la señal más directa de una variable **binaria** es `nunique() == 2`, sin importar si el `dtype` es `object`, `int64` o `bool`. 

Por eso conviene revisar `nunique()` para **todas** las columnas, no solo las categóricas.

In [16]:
#Otra alternativa para obtener datos categóricos
df_bank.select_dtypes(exclude='number')

,job,marital,education,default,housing,loan,contact,month,poutcome,y
0,unemployed,married,primary,no,no,no,cellular,oct,unknown,no
1,services,married,secondary,no,yes,yes,cellular,may,failure,no
2,management,single,tertiary,no,yes,no,cellular,apr,failure,no
3,management,married,tertiary,no,yes,yes,unknown,jun,unknown,no
4,blue-collar,married,secondary,no,yes,no,unknown,may,unknown,no
...,...,...,...,...,...,...,...,...,...,...
4516,services,married,secondary,no,yes,no,cellular,jul,unknown,no
4517,self-employed,married,tertiary,yes,yes,yes,unknown,may,unknown,no
4518,technician,married,secondary,no,no,no,cellular,aug,unknown,no
4519,blue-collar,married,secondary,no,no,no,cellular,feb,other,no


In [17]:
df_bank_categoricos['education'].nunique(), df_bank_categoricos['education'].unique()

(4, array(['primary', 'secondary', 'tertiary', 'unknown'], dtype=object))

In [18]:
df_bank_categoricos['education'].value_counts()

education
secondary    2306
tertiary     1350
primary       678
unknown       187
Name: count, dtype: int64

In [19]:
#Guardamos las columnas originales de df_bank
columns_bank = df_bank.columns
columns_bank

Index(['age', 'job', 'marital', 'education', 'default', 'balance', 'housing',
       'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays',
       'previous', 'poutcome', 'y'],
      dtype='object')

In [20]:
#Guardar las columnas de las variables cuantitivas
columns_cuantitativas = df_bank_cuantitativos.columns
columns_cuantitativas

Index(['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous'], dtype='object')

In [21]:
#Guardar las columnas de las variables categóricas
columns_categoricas = df_bank_categoricos.columns
columns_categoricas

Index(['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact',
       'month', 'poutcome', 'y'],
      dtype='object')

In [22]:
# Categóricos (Multiestado o cualitativos) --->  (nominales, ordinales)
df_bank_categoricos.head()

,job,marital,education,default,housing,loan,contact,month,poutcome,y
0,unemployed,married,primary,no,no,no,cellular,oct,unknown,no
1,services,married,secondary,no,yes,yes,cellular,may,failure,no
2,management,single,tertiary,no,yes,no,cellular,apr,failure,no
3,management,married,tertiary,no,yes,yes,unknown,jun,unknown,no
4,blue-collar,married,secondary,no,yes,no,unknown,may,unknown,no


In [23]:
# Valores únicos y categorías de la columna default
df_bank_categoricos['default'].unique(), df_bank_categoricos['default'].nunique()

(array(['no', 'yes'], dtype=object), 2)

In [24]:
# Lista de columnas y sus categorías
for col in columns_categoricas:
    print(f'>> La columna {col} tiene {df_bank_categoricos[col].nunique()} categorías y son:\n {df_bank_categoricos[col].unique()} \n')

>> La columna job tiene 12 categorías y son:
 ['unemployed' 'services' 'management' 'blue-collar' 'self-employed'
 'technician' 'entrepreneur' 'admin.' 'student' 'housemaid' 'retired'
 'unknown'] 

>> La columna marital tiene 3 categorías y son:
 ['married' 'single' 'divorced'] 

>> La columna education tiene 4 categorías y son:
 ['primary' 'secondary' 'tertiary' 'unknown'] 

>> La columna default tiene 2 categorías y son:
 ['no' 'yes'] 

>> La columna housing tiene 2 categorías y son:
 ['no' 'yes'] 

>> La columna loan tiene 2 categorías y son:
 ['no' 'yes'] 

>> La columna contact tiene 3 categorías y son:
 ['cellular' 'unknown' 'telephone'] 

>> La columna month tiene 12 categorías y son:
 ['oct' 'may' 'apr' 'jun' 'feb' 'aug' 'jan' 'jul' 'nov' 'sep' 'mar' 'dec'] 

>> La columna poutcome tiene 4 categorías y son:
 ['unknown' 'failure' 'other' 'success'] 

>> La columna y tiene 2 categorías y son:
 ['no' 'yes'] 



## 1.5 Variables Categoricas ordinales

**Las variables categóricas ordinales son variables con categorías con orden, sin distancias necesariamente iguales**

Siguiendo nuestro ejemplo, `education` representa niveles educativos con un orden razonable. Aun así, la distancia entre niveles no tiene por qué ser igual: pasar de `primary` a `secondary` no equivale necesariamente a pasar de `secondary` a `tertiary`.

`month` es otra variable categórica ordinal, esta tiene una secuencia temporal, pero es una variable cíclica: sin embargo, diciembre y enero están próximos en el calendario. 

A diferencia de obtener una variable binaria o numérica, **el código no puede detectar el orden por sí solo**: 

- `nunique()` y `unique()` muestran cuántas categorías hay y cuáles son, pero decidir que `primary < secondary < tertiary` requiere conocimiento del dominio. Por eso necesitamos declarar explícitamente la lista de variable `ordinales_cat` y el orden con `pd.CategoricalDtype(ordered=True)`.

In [25]:
# `education` tiene un orden natural; `month` es temporal y conviene tratarlo aparte.
ordinales_cat = ['education']
nominales_cat = [
    columna for columna in df_bank_categoricos.columns
    if columna not in ordinales_cat and columna != 'y'
]
binarias_cat = ['default', 'housing', 'loan', 'y']


In [26]:
binarias_cat

['default', 'housing', 'loan', 'y']

In [27]:
# dataframes con datos ordinales categóricos
df_bank_categoricos_ord = df_bank_categoricos[ordinales_cat]
df_bank_categoricos_ord.head()

,education
0,primary
1,secondary
2,tertiary
3,tertiary
4,secondary


In [28]:
# Declarar el orden evita que una transformación posterior lo invente o lo pierda.
education_order = ['primary', 'secondary', 'tertiary']
education_dtype = pd.api.types.CategoricalDtype(
    categories=education_order,
    ordered=True
)
df_bank_categoricos_ord = df_bank_categoricos[ordinales_cat].copy()
df_bank_categoricos_ord['education'] = df_bank_categoricos_ord['education'].astype(education_dtype)
df_bank_categoricos_ord['education'].dtype

CategoricalDtype(categories=['primary', 'secondary', 'tertiary'], ordered=True, categories_dtype=object)

## 1.7 Categóricas Nominales

Una vez descartadas las binarias y las ordinales, lo que queda entre las columnas de tipo `object` son **nominales**: categorías sin orden natural (`job`, `marital`, `contact`, `poutcome`, etc.).

- Aquí el criterio es por descarte: si no tiene exactamente dos valores y no le asignamos un orden explícito, es nominal.

In [29]:
nominales_cat = [
    columna for columna in df_bank_categoricos.columns
    if columna not in ordinales_cat and columna not in binarias_cat
]
nominales_cat

['job', 'marital', 'contact', 'month', 'poutcome']

In [30]:
nominales_cat

['job', 'marital', 'contact', 'month', 'poutcome']

In [31]:
# Base de datos con datos nominales categóricos
df_bank_categoricos_nom = df_bank_categoricos[nominales_cat]
df_bank_categoricos_nom


,job,marital,contact,month,poutcome
0,unemployed,married,cellular,oct,unknown
1,services,married,cellular,may,failure
2,management,single,cellular,apr,failure
3,management,married,unknown,jun,unknown
4,blue-collar,married,unknown,may,unknown
...,...,...,...,...,...
4516,services,married,cellular,jul,unknown
4517,self-employed,married,unknown,may,unknown
4518,technician,married,cellular,aug,unknown
4519,blue-collar,married,cellular,feb,other


# 2. Valores faltantes: identificar antes de imputar

Un **valor faltante (o missing value)** es la ausencia de un dato o información en una celda específica para una variable dentro de un conjunto de datos

Un valor faltante no siempre significa lo mismo: 

- Puede ser una medición no realizada, una respuesta omitida o un valor que no aplica.

Antes de decidir, necesitamos cuantificar el problema y consultar el significado de la variable.

En este cuaderno solo hacemos una introducción. La comparación detallada de métodos de imputación continúa en el módulo de **Tratamiento de datos faltantes**.

In [32]:
#Cargamos un nuevo ejemplo dataset de peliculas
df_movie = pd.read_csv('Data/movie_metadata.csv')
df_movie.head()

,color,director_name,num_critic_for_reviews,duration,director_facebook_likes,actor_3_facebook_likes,actor_2_name,actor_1_facebook_likes,gross,genres,...,num_user_for_reviews,language,country,content_rating,budget,title_year,actor_2_facebook_likes,imdb_score,aspect_ratio,movie_facebook_likes
0,Color,James Cameron,723.0,178.0,0.0,855.0,Joel David Moore,1000.0,760505847.0,Action|Adventure|Fantasy|Sci-Fi,...,3054.0,English,USA,PG-13,237000000.0,2009.0,936.0,7.9,1.78,33000
1,Color,Gore Verbinski,302.0,169.0,563.0,1000.0,Orlando Bloom,40000.0,309404152.0,Action|Adventure|Fantasy,...,1238.0,English,USA,PG-13,300000000.0,2007.0,5000.0,7.1,2.35,0
2,Color,Sam Mendes,602.0,148.0,0.0,161.0,Rory Kinnear,11000.0,200074175.0,Action|Adventure|Thriller,...,994.0,English,UK,PG-13,245000000.0,2015.0,393.0,6.8,2.35,85000
3,Color,Christopher Nolan,813.0,164.0,22000.0,23000.0,Christian Bale,27000.0,448130642.0,Action|Thriller,...,2701.0,English,USA,PG-13,250000000.0,2012.0,23000.0,8.5,2.35,164000
4,NaN,Doug Walker,NaN,NaN,131.0,NaN,Rob Walker,131.0,NaN,Documentary,...,NaN,NaN,NaN,NaN,NaN,NaN,12.0,7.1,NaN,0


In [33]:
#Obtenemos la información general del dataframe
df_movie.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5043 entries, 0 to 5042
Data columns (total 28 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   color                      5024 non-null   object 
 1   director_name              4939 non-null   object 
 2   num_critic_for_reviews     4993 non-null   float64
 3   duration                   5028 non-null   float64
 4   director_facebook_likes    4939 non-null   float64
 5   actor_3_facebook_likes     5020 non-null   float64
 6   actor_2_name               5030 non-null   object 
 7   actor_1_facebook_likes     5036 non-null   float64
 8   gross                      4159 non-null   float64
 9   genres                     5043 non-null   object 
 10  actor_1_name               5036 non-null   object 
 11  movie_title                5043 non-null   object 
 12  num_voted_users            5043 non-null   int64  
 13  cast_total_facebook_likes  5043 non-null   int64

In [35]:
df_movie.isna().sum()

color                         19
director_name                104
num_critic_for_reviews        50
duration                      15
director_facebook_likes      104
actor_3_facebook_likes        23
actor_2_name                  13
actor_1_facebook_likes         7
gross                        884
genres                         0
actor_1_name                   7
movie_title                    0
num_voted_users                0
cast_total_facebook_likes      0
actor_3_name                  23
facenumber_in_poster          13
plot_keywords                153
movie_imdb_link                0
num_user_for_reviews          21
language                      14
country                        5
content_rating               303
budget                       492
title_year                   108
actor_2_facebook_likes        13
imdb_score                     0
aspect_ratio                 329
movie_facebook_likes           0
dtype: int64

In [36]:
# missing_summary
missing_summary = df_movie.isna().agg(['sum', 'mean']).T.rename(columns = {'sum':'n_faltantes', 'mean':'proporcion_faltantes'})
missing_summary

,n_faltantes,proporcion_faltantes
color,19.0,0.003768
director_name,104.0,0.020623
num_critic_for_reviews,50.0,0.009915
duration,15.0,0.002974
director_facebook_likes,104.0,0.020623
actor_3_facebook_likes,23.0,0.004561
actor_2_name,13.0,0.002578
actor_1_facebook_likes,7.0,0.001388
gross,884.0,0.175292
genres,0.0,0.000000


In [37]:
df_movie['color']

0       Color
1       Color
2       Color
3       Color
4         NaN
        ...  
5038    Color
5039    Color
5040    Color
5041    Color
5042    Color
Name: color, Length: 5043, dtype: object

### Manejando datos Faltantes (intro)

In [38]:
# Opción 1: eliminar filas completas solo cuando la pérdida sea aceptable.
df_movie_clean = df_movie.dropna()
df_movie_clean

,color,director_name,num_critic_for_reviews,duration,director_facebook_likes,actor_3_facebook_likes,actor_2_name,actor_1_facebook_likes,gross,genres,...,num_user_for_reviews,language,country,content_rating,budget,title_year,actor_2_facebook_likes,imdb_score,aspect_ratio,movie_facebook_likes
0,Color,James Cameron,723.0,178.0,0.0,855.0,Joel David Moore,1000.0,760505847.0,Action|Adventure|Fantasy|Sci-Fi,...,3054.0,English,USA,PG-13,237000000.0,2009.0,936.0,7.9,1.78,33000
1,Color,Gore Verbinski,302.0,169.0,563.0,1000.0,Orlando Bloom,40000.0,309404152.0,Action|Adventure|Fantasy,...,1238.0,English,USA,PG-13,300000000.0,2007.0,5000.0,7.1,2.35,0
2,Color,Sam Mendes,602.0,148.0,0.0,161.0,Rory Kinnear,11000.0,200074175.0,Action|Adventure|Thriller,...,994.0,English,UK,PG-13,245000000.0,2015.0,393.0,6.8,2.35,85000
3,Color,Christopher Nolan,813.0,164.0,22000.0,23000.0,Christian Bale,27000.0,448130642.0,Action|Thriller,...,2701.0,English,USA,PG-13,250000000.0,2012.0,23000.0,8.5,2.35,164000
5,Color,Andrew Stanton,462.0,132.0,475.0,530.0,Samantha Morton,640.0,73058679.0,Action|Adventure|Sci-Fi,...,738.0,English,USA,PG-13,263700000.0,2012.0,632.0,6.6,2.35,24000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5026,Color,Olivier Assayas,81.0,110.0,107.0,45.0,Béatrice Dalle,576.0,136007.0,Drama|Music|Romance,...,39.0,French,France,R,4500.0,2004.0,133.0,6.9,2.35,171
5027,Color,Jafar Panahi,64.0,90.0,397.0,0.0,Nargess Mamizadeh,5.0,673780.0,Drama,...,26.0,Persian,Iran,Not Rated,10000.0,2000.0,0.0,7.5,1.85,697
5033,Color,Shane Carruth,143.0,77.0,291.0,8.0,David Sullivan,291.0,424760.0,Drama|Sci-Fi|Thriller,...,371.0,English,USA,PG-13,7000.0,2004.0,45.0,7.0,1.85,19000
5035,Color,Robert Rodriguez,56.0,81.0,0.0,6.0,Peter Marquardt,121.0,2040920.0,Action|Crime|Drama|Romance|Thriller,...,130.0,Spanish,USA,R,7000.0,1992.0,20.0,6.9,1.37,0


In [39]:
df_movie_clean.shape

(3755, 28)

In [40]:
#verificar que no se tengan valores faltantes
df_movie_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3755 entries, 0 to 5042
Data columns (total 28 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   color                      3755 non-null   object 
 1   director_name              3755 non-null   object 
 2   num_critic_for_reviews     3755 non-null   float64
 3   duration                   3755 non-null   float64
 4   director_facebook_likes    3755 non-null   float64
 5   actor_3_facebook_likes     3755 non-null   float64
 6   actor_2_name               3755 non-null   object 
 7   actor_1_facebook_likes     3755 non-null   float64
 8   gross                      3755 non-null   float64
 9   genres                     3755 non-null   object 
 10  actor_1_name               3755 non-null   object 
 11  movie_title                3755 non-null   object 
 12  num_voted_users            3755 non-null   int64  
 13  cast_total_facebook_likes  3755 non-null   int64  
 1

## tratar datos faltantes en columnas numéricas

In [41]:
# Para columnas numéricas
df_movie['duration']

0       178.0
1       169.0
2       148.0
3       164.0
4         NaN
        ...  
5038     87.0
5039     43.0
5040     76.0
5041    100.0
5042     90.0
Name: duration, Length: 5043, dtype: float64

In [44]:
#promedio
df_movie['duration'] = df_movie['duration'].fillna(df_movie['duration'].mean())
df_movie

,color,director_name,num_critic_for_reviews,duration,director_facebook_likes,actor_3_facebook_likes,actor_2_name,actor_1_facebook_likes,gross,genres,...,num_user_for_reviews,language,country,content_rating,budget,title_year,actor_2_facebook_likes,imdb_score,aspect_ratio,movie_facebook_likes
0,Color,James Cameron,723.0,178.000000,0.0,855.0,Joel David Moore,1000.0,760505847.0,Action|Adventure|Fantasy|Sci-Fi,...,3054.0,English,USA,PG-13,237000000.0,2009.0,936.0,7.9,1.78,33000
1,Color,Gore Verbinski,302.0,169.000000,563.0,1000.0,Orlando Bloom,40000.0,309404152.0,Action|Adventure|Fantasy,...,1238.0,English,USA,PG-13,300000000.0,2007.0,5000.0,7.1,2.35,0
2,Color,Sam Mendes,602.0,148.000000,0.0,161.0,Rory Kinnear,11000.0,200074175.0,Action|Adventure|Thriller,...,994.0,English,UK,PG-13,245000000.0,2015.0,393.0,6.8,2.35,85000
3,Color,Christopher Nolan,813.0,164.000000,22000.0,23000.0,Christian Bale,27000.0,448130642.0,Action|Thriller,...,2701.0,English,USA,PG-13,250000000.0,2012.0,23000.0,8.5,2.35,164000
4,NaN,Doug Walker,NaN,107.201074,131.0,NaN,Rob Walker,131.0,NaN,Documentary,...,NaN,NaN,NaN,NaN,NaN,NaN,12.0,7.1,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5038,Color,Scott Smith,1.0,87.000000,2.0,318.0,Daphne Zuniga,637.0,NaN,Comedy|Drama,...,6.0,English,Canada,NaN,NaN,2013.0,470.0,7.7,NaN,84
5039,Color,NaN,43.0,43.000000,NaN,319.0,Valorie Curry,841.0,NaN,Crime|Drama|Mystery|Thriller,...,359.0,English,USA,TV-14,NaN,NaN,593.0,7.5,16.00,32000
5040,Color,Benjamin Roberds,13.0,76.000000,0.0,0.0,Maxwell Moody,0.0,NaN,Drama|Horror|Thriller,...,3.0,English,USA,NaN,1400.0,2013.0,0.0,6.3,NaN,16
5041,Color,Daniel Hsia,14.0,100.000000,0.0,489.0,Daniel Henney,946.0,10443.0,Comedy|Drama|Romance,...,9.0,English,USA,PG-13,NaN,2012.0,719.0,6.3,2.35,660


In [45]:
df_movie['duration']

0       178.000000
1       169.000000
2       148.000000
3       164.000000
4       107.201074
           ...    
5038     87.000000
5039     43.000000
5040     76.000000
5041    100.000000
5042     90.000000
Name: duration, Length: 5043, dtype: float64

In [46]:
# Opción 2: imputar una columna numérica con la mediana.
# La mediana suele ser más resistente a valores extremos que el promedio.
df_movie['duration'].median()

np.float64(103.0)

## Estrategias iniciales para valores faltantes

La estrategia depende del tipo de variable y del contexto. 

- Eliminar filas solo si la pérdida es pequeña y no introduce sesgo
- Imputar con estadísticas calculadas en el conjunto de entrenamiento cuando preparemos un modelo de ML.

Para datos categorícos, `Unknown` debe distinguirse de una categoría real. 

En un proyecto de ciencia de datos, la recomendación es documentar cuántos valores fueron imputados y por qué.

In [47]:
df_movie['color'].unique()

array(['Color', nan, ' Black and White'], dtype=object)

## 3. Reporte de calidad de datos: Automatizar la clasificación

Repetir manualmente los pasos anteriores para cada columna es lento y propenso a errores en datasets con muchas variables. 

Podemos convertir la secuencia de preguntas de la sección 1.1.1 en una **función reutilizable** que recorra un `DataFrame` y devuelva otro `DataFrame` con:

- El tipo almacenado (`dtype`) y el tipo sugerido (binaria, categórica nominal, categórica ordinal, cuantitativa discreta o continua).
- Indicadores de calidad: número de valores únicos, número y porcentaje de valores faltantes.
- Un ejemplo de los valores observados, útil para auditar rápidamente el resultado.

La función usa heurísticas (número de valores únicos, `dtype`) para proponer un tipo, pero deja como **parámetros** las columnas que el analista ya sabe que son binarias u ordinales, porque esa información depende del significado del negocio y no puede inferirse solo del código.


In [ ]:
class Numeros():
    def __init__(x=0, y=0):
        self.x=x
        self.y=y
    def suma(self):
        self.suma = self.x+self.y
        

In [60]:
def reporte_calidad_datos(df, ordinales=None, binarias=None, max_categorias_discretas=20):
    """Clasifica cada columna de df y arma un reporte de calidad de datos.

    Parameters
    ----------
    df : pd.DataFrame
        Datos a diagnosticar.
    ordinales : list[str], opcional
        Columnas que el analista ya identificó como categóricas ordinales.
    binarias : list[str], opcional
        Columnas que el analista ya identificó como binarias (además de las
        que se detectan automáticamente por tener 2 valores únicos).
    max_categorias_discretas : int
        Umbral de valores únicos para distinguir una cuantitativa discreta
        (pocos valores enteros distintos) de una continua.

    Returns
    -------
    pd.DataFrame
        Una fila por variable con su clasificación y métricas de calidad.
    """

    ordinales = set(ordinales or [])
    binarias = set(binarias or [])
    n_filas = len(df)
    filas_reporte = []

    for col in df.columns:
        serie = df[col]

        n_unicos = serie.nunique(dropna=True)
        n_faltantes = int(serie.isna().sum())
        pct_faltantes = round(100*n_faltantes/n_filas, 2) if n_filas else 0.0
        es_numerica = pd.api.types.is_numeric_dtype(serie)

        if col in binarias or n_unicos == 2:
            tipo_sugerida = 'Binaria'
        elif col in ordinales:
            tipo_sugerida = 'Categoríca Ordinal'
        elif es_numerica:
            es_entera = pd.api.types.is_integer_dtype(serie)
            if es_entera and n_unicos <= max_categorias_discretas:
                tipo_sugerida = 'Cuantitativa Discreta'
            else: 
                tipo_sugerida = 'Cuantitativa Continua'
        else: 
            tipo_sugerida = 'Categoríca Nominal'
            
        filas_reporte.append({'variable': col,
                            'dtype': str(serie.dtype),
                            'valores_unicos': n_unicos,
                            'Valores faltantes':n_faltantes,
                            'pct Valores faltantes': pct_faltantes,
                            'es_numerica': es_numerica,
                             'Clasificación Sugerida': tipo_sugerida,
                             #'Advertencia': advertencia,
                             #'Categorias_unicas': cat_unique
                             })

    reporte = pd.DataFrame(filas_reporte)
    return reporte


In [61]:
df_bank

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,30,unemployed,married,primary,no,1787,no,no,cellular,19,oct,79,1,-1,0,unknown,no
1,33,services,married,secondary,no,4789,yes,yes,cellular,11,may,220,1,339,4,failure,no
2,35,management,single,tertiary,no,1350,yes,no,cellular,16,apr,185,1,330,1,failure,no
3,30,management,married,tertiary,no,1476,yes,yes,unknown,3,jun,199,4,-1,0,unknown,no
4,59,blue-collar,married,secondary,no,0,yes,no,unknown,5,may,226,1,-1,0,unknown,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4516,33,services,married,secondary,no,-333,yes,no,cellular,30,jul,329,5,-1,0,unknown,no
4517,57,self-employed,married,tertiary,yes,-3313,yes,yes,unknown,9,may,153,1,-1,0,unknown,no
4518,57,technician,married,secondary,no,295,no,no,cellular,19,aug,151,11,-1,0,unknown,no
4519,28,blue-collar,married,secondary,no,1137,no,no,cellular,6,feb,129,4,211,3,other,no


In [63]:
reporte_calidad_datos(df_bank, ordinales=['education'], max_categorias_discretas=20)

,variable,dtype,valores_unicos,Valores faltantes,pct Valores faltantes,es_numerica,Clasificación Sugerida
0,age,int64,67,0,0.0,True,Cuantitativa Continua
1,job,object,12,0,0.0,False,Categoríca Nominal
2,marital,object,3,0,0.0,False,Categoríca Nominal
3,education,object,4,0,0.0,False,Categoríca Ordinal
4,default,object,2,0,0.0,False,Binaria
5,balance,int64,2353,0,0.0,True,Cuantitativa Continua
6,housing,object,2,0,0.0,False,Binaria
7,loan,object,2,0,0.0,False,Binaria
8,contact,object,3,0,0.0,False,Categoríca Nominal
9,day,int64,31,0,0.0,True,Cuantitativa Continua


### Aplicando el reporte a `bank.csv`

Le pasamos las columnas que ya sabemos que son ordinales (`education`) y binarias (`default`, `housing`, `loan`, `y`); el resto se clasifica automáticamente.


### Aplicando el reporte a `movie_metadata.csv`

Sin pasar `ordinales` ni `binarias`, la función solo puede basarse en heurísticas: detecta binarias por conteo de categorías y separa numéricas de nominales, pero no puede saber si alguna columna tiene un orden. Comparar este resultado con lo que ya sabes del dataset ayuda a detectar columnas que requieren revisión manual.


In [67]:

reporte_movie = reporte_calidad_datos(df_movie)
reporte_movie

,variable,dtype,valores_unicos,Valores faltantes,pct Valores faltantes,es_numerica,Clasificación Sugerida
0,color,object,2,19,0.38,False,Binaria
1,director_name,object,2398,104,2.06,False,Categoríca Nominal
2,num_critic_for_reviews,float64,528,50,0.99,True,Cuantitativa Continua
3,duration,float64,192,0,0.00,True,Cuantitativa Continua
4,director_facebook_likes,float64,435,104,2.06,True,Cuantitativa Continua
5,actor_3_facebook_likes,float64,906,23,0.46,True,Cuantitativa Continua
6,actor_2_name,object,3032,13,0.26,False,Categoríca Nominal
7,actor_1_facebook_likes,float64,878,7,0.14,True,Cuantitativa Continua
8,gross,float64,4035,884,17.53,True,Cuantitativa Continua
9,genres,object,914,0,0.00,False,Categoríca Nominal


In [68]:
reporte_movie[reporte_movie['Clasificación Sugerida'] == 'Categoríca Nominal']

,variable,dtype,valores_unicos,Valores faltantes,pct Valores faltantes,es_numerica,Clasificación Sugerida
1,director_name,object,2398,104,2.06,False,Categoríca Nominal
6,actor_2_name,object,3032,13,0.26,False,Categoríca Nominal
9,genres,object,914,0,0.00,False,Categoríca Nominal
10,actor_1_name,object,2097,7,0.14,False,Categoríca Nominal
11,movie_title,object,4917,0,0.00,False,Categoríca Nominal
14,actor_3_name,object,3521,23,0.46,False,Categoríca Nominal
16,plot_keywords,object,4760,153,3.03,False,Categoríca Nominal
17,movie_imdb_link,object,4919,0,0.00,False,Categoríca Nominal
19,language,object,46,14,0.28,False,Categoríca Nominal
20,country,object,65,5,0.10,False,Categoríca Nominal


## Práctica de Laboratorio: Clasificación de tipos de Datos

Responde y justifica tus decisiones. No existe una única respuesta correcta si explicas el criterio.

1. Clasifica `job`, `education`, `month`, `campaign` y `y` según su significado estadístico.
2. ¿Por qué no sería correcto calcular el promedio de `job` aunque se codifique con números?
3. ¿Qué problema puede aparecer si se codifica `month` como `1, 2, ..., 12` y se usa esa columna directamente en un modelo?
4. Calcula la proporción de personas con `y == 'yes'` y compárala por nivel de `education`.
5. Elige entre eliminar o imputar los faltantes de `movie_metadata.csv`. Reporta cuántas filas o valores afecta tu decisión.
6. Ejecuta `reporte_calidad_datos(df_movie)` y revisa el resultado: ¿hay alguna columna cuyo `tipo_sugerido` te parezca incorrecto? Explica por qué la heurística falla en ese caso.

### Reto

Modifica `reporte_calidad_datos` para que agregue una columna `advertencia` que marque con `True` las variables con más de 30% de valores faltantes.